In [ ]:
import torch
import numpy as np
from tqdm import tqdm
import yaml
import sys

project_root = r""
sys.path.append(project_root)

from utils import create_model
from dataset import  DiffractionDataset

In [ ]:
CONFIG_PATH = r"" 
MODEL_PATH = r""
HDF5_PATH = r""

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

PARAM_NAMES = ["dist", "poni1", "poni2", "rot1", "rot2", "rot3"]
PARAM_UNITS = ["m", "m", "m", "rad", "rad", "rad"]

In [ ]:
# Load model in inference mode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = create_model(config)

print(f"Loading model weights from {MODEL_PATH}")
state_dict = torch.load(MODEL_PATH, map_location=device)

if any(k.startswith('_orig_mod.') for k in state_dict.keys()):
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k[10:] if k.startswith('_orig_mod.') else k
        new_state_dict[name] = v
    state_dict = new_state_dict

model.load_state_dict(state_dict)
model.to(device)

print("Model loaded.")

In [ ]:
def benchmark_latency(model, device, input_shape=(1, 3, 1056, 1056), warmup_steps=50, num_runs=300):
    """
    Rigorously measures the inference latency of a PyTorch model.
    """
    print(f"Benchmarking on: {device}")
    model.eval()
    
    # Create a dummy tensor that matches your exact production input
    dummy_input = torch.randn(input_shape, dtype=torch.float32).to(device)
    
    # Initialize CUDA timing events
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    # --- PHASE 1: WARM-UP ---
    # The first few passes are always slow due to memory allocation 
    # and cuDNN benchmarking. We must discard these.
    print(f"Performing {warmup_steps} warm-up iterations...")
    with torch.no_grad():
        for _ in range(warmup_steps):
            _ = model(dummy_input)
            
    # Force the CPU to wait until all warm-up tasks on the GPU are finished
    torch.cuda.synchronize()
    
    # --- PHASE 2: MEASUREMENT ---
    print(f"Measuring latency over {num_runs} iterations...")
    timings = []
    
    with torch.no_grad():
        for _ in tqdm(range(num_runs)):
            # Record start time
            start_event.record()
            
            # Forward pass
            _ = model(dummy_input)
            
            # Record end time
            end_event.record()
            
            # Synchronize to ensure the forward pass is completely finished
            torch.cuda.synchronize()
            
            # Calculate elapsed time in milliseconds
            elapsed_time_ms = start_event.elapsed_time(end_event)
            timings.append(elapsed_time_ms)
            
    # --- PHASE 3: STATISTICAL ANALYSIS ---
    timings = np.array(timings)
    mean_latency = np.mean(timings)
    std_latency = np.std(timings)
    percentile_99 = np.percentile(timings, 99)
    
    # Calculate throughput (Frames Per Second)
    fps = 1000.0 / mean_latency
    
    print("\n--- Benchmarking Results ---")
    print(f"Batch Size:      {input_shape[0]}")
    print(f"Mean Latency:    {mean_latency:.2f} ms")
    print(f"Std Deviation:   ±{std_latency:.2f} ms")
    print(f"99th Percentile: {percentile_99:.2f} ms (Worst-case guarantee)")
    print(f"Throughput:      {fps:.2f} frames/second")
    
    return mean_latency, fps

# Run the benchmark
# Assuming 'model' and 'device' are already defined in your notebook
mean_time, throughput = benchmark_latency(model, device)